# 07 — Field-photo YOLOv8 evaluation (FROZEN — NO RETRAINING)

Loads the existing `models/crop_health_best.pt` classifier, runs it across the labelled ESP32 photo folder, and reports per-class precision/recall + the SHA-256 audit hash.

**Read-only**: this notebook never writes a new `.pt` file.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from visual_validation import config
from visual_validation.models.field_yolov8 import FieldYOLOv8Predictor
config.ensure_dirs()
yolo = FieldYOLOv8Predictor()
print('sha256 =', yolo.model_sha256)

In [ ]:
from pathlib import Path
TEST_DIR = Path('data/visual/field_photos/labelled')
df = yolo.predict_folder(TEST_DIR)
df.head()

In [ ]:
# If filename pattern encodes the ground truth (e.g. <class>_<n>.jpg) extract it
import re
def parse_gt(p):
    name = Path(p).stem
    m = re.match(r'([a-z_]+)_', name)
    return m.group(1) if m else None
df['gt_raw'] = df['image_path'].map(parse_gt)
df['gt_harmonized'] = df['gt_raw'].map(lambda c: config.YOLOV8_TO_HARMONIZED.get(c))
df.head()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_true = df['gt_harmonized'].dropna().tolist()
y_pred = df.loc[df['gt_harmonized'].notna(), 'harmonized_class'].tolist()
print(classification_report(y_true, y_pred, zero_division=0))
print('confusion_matrix:'); print(confusion_matrix(y_true, y_pred))